In [8]:
import ccdproc
import numpy as np
from astropy import units as u


In [9]:
#Bias
bias_path=r"C:\Users\Lenovo\Desktop\laby_fitsy\Bias"
bias=ccdproc.ImageFileCollection(bias_path)
master_bias = ccdproc.combine(
    bias.ccds(imagetyp='BIAS', ccd_kwargs={'unit': 'adu'}),
    method='average',
    sigma_clip=True
)


In [11]:
#Dark
dark_path=r"C:\Users\Lenovo\Desktop\laby_fitsy\Dark"
darks=ccdproc.ImageFileCollection(dark_path)
calibarated_darks = [
    ccdproc.subtract_bias(ccd, master_bias)
    for ccd in darks.ccds(imagetyp='DARK', ccd_kwargs={'unit': 'adu'})
]


master_dark = ccdproc.combine(
    calibarated_darks,
    method='average',
    sigma_clip=True
)

In [12]:
#Flat
flat_path=r"C:\Users\Lenovo\Desktop\laby_fitsy\Flat"
flats=ccdproc.ImageFileCollection(flat_path)
filters=flats.values('filter', unique=True)
master_flats=[]
for filter in filters:
    calibrated_flats=[]
    for ccd in flats.ccds(imagetyp='FLAT', filter=filter, ccd_kwargs={'unit': 'adu'}):
        ccd_clean = ccdproc.ccd_process(
            ccd,
            master_bias=master_bias,
            dark_frame=master_dark,
            exposure_key='EXPTIME',
            exposure_unit=u.second,
            dark_scale=True
        )
        calibrated_flats.append(ccd_clean)
    master_flat = ccdproc.combine(
        calibrated_flats,
        method='average',
        scale=lambda x: 1.0 / np.median(x),
        sigma_clip=True
    )
    master_flats.append(master_flat)

master_flats


[CCDData([[-9.10695561e-04,  6.61879130e-05, -8.59395002e-05, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [-5.43941864e-05, -5.01660664e-05, -3.14081140e-05, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [-6.59731535e-05,  9.19800346e-05,  1.77588413e-05, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          ...,
          [ 1.78209560e-05,  5.07068085e-06, -9.05910308e-06, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [ 3.89119486e-05,  9.18851944e-06,  5.99039666e-05, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [ 9.37732207e-05, -2.03816472e-05,  2.25517983e-05, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
         shape=(4210, 6280), unit='adu'),
 CCDData([[ 7.84504782e-05,  2.57229309e-04, -3.76956009e-05, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [-9.99716552e-05, -3.0

In [ ]:
master_bias.write(r"C:\Users\Lenovo\Desktop\laby_fitsy\Masters\master_bias.fits", overwrite=True)
master_dark.write(r"C:\Users\Lenovo\Desktop\laby_fitsy\Masters\master_dark.fits", overwrite=True)
#for master in master_flats:
